In [ ]:
#!{sys.executable} -m pip install bioscan-dataset


In [ ]:
from bioscan_dataset import BIOSCAN5M

ROOT = "./bioscan"
ALL_SPLITS = "pretrain+train+val+test+key_unseen+val_unseen+test_unseen"

dataset = BIOSCAN5M(
    root=ROOT,
    split=ALL_SPLITS,
    modality="metadata",
    download=True   # set False if already downloaded
)

df = dataset.metadata.reset_index(drop=True)

print("rows:", len(df))
print("df index range:", df.index.min(), "to", df.index.max())
print(df["split"].value_counts())


In [ ]:
# Orders allowed for Insecta
insect_orders = [
    "Lepidoptera", "Psocodea", "Odonata", "Siphonaptera", "Zoraptera",
    "Phasmatodea", "Zygentoma", "Thysanoptera", "Strepsiptera",
    "Archaeognatha", "Dermaptera", "Raphidioptera",
    "Diptera", "Coleoptera", "Hymenoptera", "Hemiptera",
    "Orthoptera", "Trichoptera", "Neuroptera"
]

# Orders allowed for Arachnida
arachnid_orders = [
    "Araneae",
    "Opiliones"
]


In [ ]:
mask = (
    ((df["class"] == "Insecta") & df["order"].isin(insect_orders)) |
    ((df["class"] == "Arachnida") & df["order"].isin(arachnid_orders)) |
    (df["class"].isin(["Diplopoda", "Chilopoda"]))
)

filtered = df[mask].copy()

# preserve true positional row index for image loading later
filtered["source_idx"] = filtered.index

print("Total filtered specimens:", len(filtered))
print("\nClass counts:")
print(filtered["class"].value_counts())

print("\nOrder counts:")
print(filtered["order"].value_counts())


In [ ]:
filtered.to_csv("bioscan_selected_classes_orders_full.csv", index=False)
print("Saved: bioscan_selected_classes_orders_full.csv")


In [ ]:
N = 20000

sampled = (
    filtered
    .groupby("order", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), N), random_state=42))
    .reset_index(drop=True)
)

print("Balanced sampled size:", len(sampled))
print("\nBalanced order counts:")
print(sampled["order"].value_counts())


In [ ]:
sampled.to_csv("selected_classes_original_20000.csv", index=False)
print("Saved: selected_classes_original_20000.csv")


In [ ]:
cols = [
    "source_idx", "processid", "class", "order", "family",
    "country", "split", "chunk", "image_path"
]

sampled[cols].to_csv("bioscan_balanced_minimal_original.csv", index=False)
print("Saved: bioscan_balanced_minimal_original.csv")


In [ ]:
summary = (
    sampled.groupby(["class", "order"])
    .size()
    .reset_index(name="count")
    .sort_values(["class", "count"], ascending=[True, False])
)

summary


In [ ]:
print("df length:", len(df))
print("df index range:", df.index.min(), df.index.max())
print("sampled source_idx range:", sampled["source_idx"].min(), sampled["source_idx"].max())
print("All sampled indices valid:", sampled["source_idx"].max() < len(df))


In [ ]:
import os
from tqdm import tqdm
from bioscan_dataset import BIOSCAN5M
import pandas as pd

OUT_DIR = "./bioscan_balanced_images_20000_original"
os.makedirs(OUT_DIR, exist_ok=True)

dataset_img = BIOSCAN5M(
    root=ROOT,
    split=ALL_SPLITS,
    modality="image",
    image_package="original_full",
    download=False
)

print("Metadata rows:", len(df))
print("Image rows:", len(dataset_img.metadata))
print("Max sampled source_idx:", sampled["source_idx"].max())

failed = []

for _, row in tqdm(sampled.iterrows(), total=len(sampled)):
    idx = int(row["source_idx"])
    pid = row["processid"]

    try:
        img, target = dataset_img[idx]
        img.save(os.path.join(OUT_DIR, f"{pid}.jpg"), quality=95)
    except Exception as e:
        failed.append({
            "source_idx": idx,
            "processid": pid,
            "error": repr(e)
        })
        print("Failed idx:", idx, "processid:", pid, "error:", repr(e))

print("Saved to:", OUT_DIR)
print("Saved images:", len(sampled) - len(failed))
print("Failed:", len(failed))

if failed:
    pd.DataFrame(failed).to_csv("bioscan_balanced_image_failures.csv", index=False)
    print("Saved failures: bioscan_balanced_image_failures.csv")
